In [1]:
import torch
import time
import pandas as pd

from torchvision.models import mobilenet_v2

In [2]:
model = mobilenet_v2(weights="DEFAULT")
model.eval()

print("Model loaded.")

Model loaded.


In [3]:
input_tensor = torch.randn(1, 3, 224, 224)

In [4]:
def benchmark(model, device, input_tensor, runs=100):

    model = model.to(device)
    input_tensor = input_tensor.to(device)

    # Warm-up runs
    for _ in range(10):
        with torch.no_grad():
            _ = model(input_tensor)

    # GPU synchronization
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    avg_latency = (end - start) / runs

    return avg_latency

In [5]:
cpu_latency = benchmark(model, "cpu", input_tensor)

print(f"CPU Average Latency: {cpu_latency:.6f} seconds")

CPU Average Latency: 0.053954 seconds


In [6]:
gpu_latency = benchmark(model, "cuda", input_tensor)

print(f"GPU Average Latency: {gpu_latency:.6f} seconds")

GPU Average Latency: 0.018986 seconds


In [7]:
results = pd.DataFrame({
    "Device": ["CPU", "GPU"],
    "Latency_Seconds": [cpu_latency, gpu_latency]
})

results.to_csv("baseline_results.csv", index=False)

print(results)
print("\nResults saved to baseline_results.csv")

  Device  Latency_Seconds
0    CPU         0.053954
1    GPU         0.018986

Results saved to baseline_results.csv
